# Projeto 1 — O que faz um aluno ir bem na prova?

**C111/C11 – Análise de Dados** · Dataset: [Student Performance Factors](https://www.kaggle.com/datasets/lainguyn123/student-performance-factors) (Kaggle) · 6.607 alunos, 20 colunas

**Objetivo:** descobrir quais fatores mais pesam na nota (`Exam_Score`) para orientar onde escolas, famílias e governo devem investir.

1. A frequência às aulas influencia a nota?
2. Mais horas de estudo dão nota maior?
3. Qual o perfil dos 10% melhores x 10% piores?
4. Renda e acesso a recursos criam desigualdade?
5. Envolvimento e escolaridade dos pais fazem diferença?
6. Quais fatores numéricos mais se relacionam com a nota?
7. Tutoria compensa?
8. Motivação, professor ou colegas: qual pesa mais?
9. Escola privada vai melhor que pública?
10. Sono e atividade física afetam a nota?

## Dados

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.figsize': (10, 5), 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False})
AZUL, LARANJA, CINZA = '#2b6cb0', '#dd6b20', '#718096'

caminho_dados = Path('dados/StudentPerformanceFactors.csv')
if not caminho_dados.is_file():
    caminho_dados = Path('trabalho') / caminho_dados
if not caminho_dados.is_file():
    raise FileNotFoundError(f'Arquivo de dados não encontrado: {caminho_dados.resolve()}')

df = pd.read_csv(caminho_dados)
print(f'Dataset carregado: {caminho_dados} | {df.shape[0]:,} linhas e {df.shape[1]} colunas')
df.head()

In [ ]:
df.describe().round(2)

## Limpeza

- Ausentes em 3 colunas (< 1,5%): preenchidos com a moda
- Nota 101: limitada a 100
- Categorias Low/Medium/High: convertidas para ordenadas

In [ ]:
print(df.isna().sum()[lambda s: s > 0])
print('Notas acima de 100:', (df['Exam_Score'] > 100).sum())

In [ ]:
for coluna in ['Teacher_Quality', 'Parental_Education_Level', 'Distance_from_Home']:
    df[coluna] = df[coluna].fillna(df[coluna].mode()[0])
df['Exam_Score'] = df['Exam_Score'].clip(upper=100)

ordens = {
    'Parental_Involvement': ['Low', 'Medium', 'High'],
    'Access_to_Resources': ['Low', 'Medium', 'High'],
    'Motivation_Level': ['Low', 'Medium', 'High'],
    'Family_Income': ['Low', 'Medium', 'High'],
    'Teacher_Quality': ['Low', 'Medium', 'High'],
    'Peer_Influence': ['Negative', 'Neutral', 'Positive'],
    'Parental_Education_Level': ['High School', 'College', 'Postgraduate'],
    'Distance_from_Home': ['Near', 'Moderate', 'Far'],
}
for coluna, ordem in ordens.items():
    df[coluna] = pd.Categorical(df[coluna], categories=ordem, ordered=True)

## Distribuição da nota

In [ ]:
media = df['Exam_Score'].mean()
plt.hist(df['Exam_Score'], bins=np.arange(54.5, 101.5, 1), color=AZUL, edgecolor='white')
plt.axvline(media, color=LARANJA, linestyle='--', label=f'Média = {media:.1f}')
plt.title('Distribuição das notas')
plt.xlabel('Nota')
plt.legend()
plt.show()

Notas concentradas (desvio de 3,9): diferenças de 1 ponto entre grupos já são relevantes.

In [ ]:
def barras_media(coluna, titulo, ax, cor=AZUL):
    medias = df.groupby(coluna, observed=True)['Exam_Score'].mean()
    barras = ax.bar(medias.index.astype(str), medias, color=cor)
    ax.bar_label(barras, fmt='%.1f', padding=3)
    ax.set_ylim(0, 100)
    ax.set_title(titulo)
    ax.set_ylabel('Nota média')
    return medias


def dispersao_e_faixas(coluna, bins, rotulos, nome):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    coef = np.polyfit(df[coluna], df['Exam_Score'], 1)
    x = np.linspace(df[coluna].min(), df[coluna].max(), 100)
    ax1.scatter(df[coluna], df['Exam_Score'], alpha=0.15, s=12, color=AZUL)
    ax1.plot(x, np.polyval(coef, x), color=LARANJA, linewidth=2)
    ax1.set_title(f'{nome} x nota')
    ax1.set_xlabel(nome)
    ax1.set_ylabel('Nota')
    faixas = pd.cut(df[coluna], bins=bins, labels=rotulos)
    medias = barras_media(faixas, 'Nota média por faixa', ax2)
    ax2.set_xlabel(nome)
    plt.tight_layout()
    plt.show()
    r = df[coluna].corr(df['Exam_Score'])
    print(f'r = {r:.2f} | +{coef[0]:.2f} ponto por unidade | '
          f'{medias.iloc[-1] - medias.iloc[0]:.1f} pontos entre a última e a primeira faixa')

## 1. Frequência

In [ ]:
dispersao_e_faixas('Attendance', [59, 70, 80, 90, 100], ['60-70%', '71-80%', '81-90%', '91-100%'], 'Frequência (%)')

**Resposta:** Principal fator (r = 0,58): quem vai a mais de 90% das aulas tira ~6 pontos a mais que quem vai a até 70%.

## 2. Horas de estudo

In [ ]:
dispersao_e_faixas('Hours_Studied', [0, 10, 15, 20, 25, 30, 44], ['1-10', '11-15', '16-20', '21-25', '26-30', '31+'], 'Horas de estudo por semana')

**Resposta:** Cada hora semanal soma ~0,3 ponto, sem platô (r = 0,45).

## 3. Grupos nos percentis inferior e superior

Os cortes P10 e P90 incluem todos os alunos empatados na nota de corte; por isso, os grupos podem não ter exatamente 10% da amostra.

In [ ]:
p10 = df['Exam_Score'].quantile(0.1)
p90 = df['Exam_Score'].quantile(0.9)
base = df[df['Exam_Score'] <= p10]
top = df[df['Exam_Score'] >= p90]

print(f'Grupo inferior (nota <= P10 = {p10:.1f}): {len(base):,} alunos ({len(base) / len(df):.1%})')
print(f'Grupo superior (nota >= P90 = {p90:.1f}): {len(top):,} alunos ({len(top) / len(df):.1%})')

caracteristicas = {
    'Pais muito envolvidos': lambda d: d['Parental_Involvement'] == 'High',
    'Muito acesso a recursos': lambda d: d['Access_to_Resources'] == 'High',
    'Colegas positivos': lambda d: d['Peer_Influence'] == 'Positive',
    'Alta motivação': lambda d: d['Motivation_Level'] == 'High',
    'Deficiência de aprendizagem': lambda d: d['Learning_Disabilities'] == 'Yes',
    'Mora longe da escola': lambda d: d['Distance_from_Home'] == 'Far',
}
perfil = pd.DataFrame({
    'Grupo inferior (<= P10)': {nome: f(base).mean() * 100 for nome, f in caracteristicas.items()},
    'Grupo superior (>= P90)': {nome: f(top).mean() * 100 for nome, f in caracteristicas.items()},
})
habitos = pd.DataFrame({'Grupo inferior (<= P10)': base[['Hours_Studied', 'Attendance']].mean(),
                        'Grupo superior (>= P90)': top[['Hours_Studied', 'Attendance']].mean()})
habitos.index = ['Horas de estudo/semana', 'Frequência (%)']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
habitos.plot.bar(ax=ax1, color=[CINZA, AZUL], rot=0, title='Hábitos')
perfil.plot.barh(ax=ax2, color=[CINZA, AZUL], title='% de alunos com a característica')
ax2.invert_yaxis()
for ax in (ax1, ax2):
    for barras in ax.containers:
        ax.bar_label(barras, fmt='%.0f', padding=3)
plt.tight_layout()
plt.show()

**Resposta:** O grupo superior apresenta frequência e horas de estudo médias maiores, além de maior proporção de alunos com alto envolvimento parental e acesso a recursos. Os tamanhos reais dos grupos são impressos acima; a comparação mostra associação, não causalidade.

## 4. Renda e acesso a recursos

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
barras_media('Family_Income', 'Renda familiar', axes[0])
barras_media('Access_to_Resources', 'Acesso a recursos', axes[1])
barras_media('Internet_Access', 'Acesso à internet', axes[2], cor=CINZA)
plt.tight_layout()
plt.show()

comparacao_renda_recursos = df.pivot_table(
    index='Family_Income', columns='Access_to_Resources', values='Exam_Score',
    aggfunc=['mean', 'count'], observed=True,
)
comparacao_renda_recursos.round(1)

**Resposta:** Na comparação descritiva, alunos de baixa renda com alto acesso a recursos têm média próxima à de alunos de alta renda com baixo acesso. A tabela mostra também o tamanho de cada grupo; essa comparação não controla os demais fatores e não permite concluir causalidade.

## 5. Pais

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
barras_media('Parental_Involvement', 'Envolvimento dos pais', ax1)
barras_media('Parental_Education_Level', 'Escolaridade dos pais', ax2, cor=CINZA)
plt.tight_layout()
plt.show()

comparacao_pais = df.pivot_table(
    index='Parental_Involvement', columns='Parental_Education_Level', values='Exam_Score',
    aggfunc=['mean', 'count'], observed=True,
)
comparacao_pais.round(1)

**Resposta:** As médias variam entre os níveis de envolvimento e escolaridade dos pais. A tabela apresenta o número de alunos em cada combinação; diferenças observadas são associações e não demonstram efeito causal.

## 6. Correlações

In [ ]:
ranking = df.select_dtypes('number').corr()['Exam_Score'].drop('Exam_Score').sort_values()
barras = plt.barh(ranking.index, ranking, color=[LARANJA if v == ranking.max() else AZUL for v in ranking])
plt.bar_label(barras, fmt='%.2f', padding=3)
plt.xlim(-0.1, ranking.max() + 0.1)
plt.title('Correlação de cada fator com a nota')
plt.show()

**Resposta:** Frequência e horas de estudo apresentam as maiores correlações lineares com a nota entre as variáveis numéricas mostradas. Correlação não implica causalidade.

## 7. Tutoria

In [ ]:
sessoes = df['Tutoring_Sessions'].clip(upper=5).astype(str).replace({'5': '5+'})
barras_media(sessoes, 'Nota média por sessões de tutoria/mês', plt.gca())
plt.show()
df.groupby(sessoes, observed=True)['Exam_Score'].agg(['mean', 'count']).round(1)

**Resposta:** As médias aumentam entre algumas faixas de sessões de tutoria, mas os tamanhos dos grupos variam (veja a tabela). O padrão é uma associação observacional, não prova que tutoria, isoladamente, cause aumento na nota.

## 8. Motivação, professor e colegas

In [ ]:
fatores = [('Motivation_Level', 'Motivação'), ('Teacher_Quality', 'Qualidade do professor'),
           ('Peer_Influence', 'Influência dos colegas')]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (coluna, nome) in zip(axes, fatores):
    medias = barras_media(coluna, nome, ax)
    print(f'{nome}: {medias.max() - medias.min():.2f} ponto entre o pior e o melhor nível')
plt.tight_layout()
plt.show()

**Resposta:** Efeitos parecidos (~1 ponto entre o pior e o melhor nível).

## 9. Pública x privada

In [ ]:
grupos = [df.loc[df['School_Type'] == t, 'Exam_Score'] for t in ['Public', 'Private']]
plt.boxplot(grupos, tick_labels=['Pública', 'Privada'], patch_artist=True,
            boxprops={'facecolor': '#bee3f8'}, medianprops={'color': LARANJA, 'linewidth': 2})
plt.title('Notas por tipo de escola')
plt.show()
print(f'Média pública: {grupos[0].mean():.2f} (n={len(grupos[0]):,}) | '
      f'privada: {grupos[1].mean():.2f} (n={len(grupos[1]):,})')

**Resposta:** As médias são muito próximas nesta amostra. A diferença descritiva, sem controle por outros fatores, não permite concluir que o tipo de escola cause diferença nas notas.

## 10. Sono e atividade física

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
barras_media('Sleep_Hours', 'Horas de sono por noite', ax1)
barras_media('Physical_Activity', 'Atividade física (h/semana)', ax2, cor=CINZA)
plt.tight_layout()
plt.show()
r_sono = df['Sleep_Hours'].corr(df['Exam_Score'])
r_atividade = df['Physical_Activity'].corr(df['Exam_Score'])
print(f'r sono = {r_sono:.2f} | r atividade = {r_atividade:.2f}')

**Resposta:** As correlações lineares de sono e atividade física com a nota são próximas de zero nesta amostra. Isso não descarta outras relações nem permite concluir causalidade.

## Conclusão

| # | Pergunta | Resposta curta |
|---|----------|----------------|
| 1 | Frequência | Maior frequência está associada a notas médias mais altas |
| 2 | Horas de estudo | Mais horas de estudo estão associadas a notas médias mais altas no intervalo observado |
| 3 | Perfil por percentis | O grupo superior apresenta mais frequência, estudo e apoio, considerando os empates nos cortes P10/P90 |
| 4 | Renda e recursos | Há diferenças entre grupos; as médias não isolam o efeito de renda ou recursos |
| 5 | Pais | Envolvimento e escolaridade apresentam associações descritivas com as notas |
| 6 | Fatores numéricos | Frequência (0,58) e estudo (0,45) dominam; sono e atividade física ~0 |
| 7 | Tutoria | As médias variam conforme as sessões; grupos têm tamanhos diferentes e a análise não prova causalidade |
| 8 | Motivação, professor, colegas | Efeitos parecidos (~1 ponto); colegas pesam tanto quanto professor |
| 9 | Pública x privada | As médias são muito próximas nesta amostra |
| 10 | Sono e atividade física | Sem efeito relevante na nota |

**Recomendação:** considerar frequência, rotina de estudo, envolvimento da família e acesso a recursos em novas investigações e ações-piloto, avaliando os resultados antes de atribuir causalidade.

*Dados observacionais e parcialmente sintéticos: mostram associação, não causalidade.*